In [1]:
# ── Cell 1: Imports & Data Transforms ─────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
import time
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

IMG_SIZE = 224

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# FIX: Removed duplicated/broken dataset lines
train_dataset = datasets.ImageFolder(
    root=r"C:\Users\ACER\Desktop\rice_datasets\train",
    transform=train_transforms
)
val_dataset = datasets.ImageFolder(
    root=r"C:\Users\ACER\Desktop\rice_datasets\val",
    transform=test_val_transforms
)

print("Classes found:", train_dataset.classes)
print(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")

Classes found: ['ClassA-Drought', 'ClassB-PestInfestation', 'ClassC-Healthy']
Train samples: 1773 | Val samples: 1773


In [2]:
# ── Cell 2: DataLoaders ────────────────────────────────────────────────
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

# Quick sanity check
for images, labels in train_loader:
    print("Image batch shape:", images.shape)  # torch.Size([32, 3, 224, 224])
    print("Labels batch shape:", labels.shape) # torch.Size([32])
    break

Image batch shape: torch.Size([32, 3, 224, 224])
Labels batch shape: torch.Size([32])


In [3]:
# ── Cell 3: Load Pretrained ResNet18 ──────────────────────────────────
NUM_CLASSES = len(train_dataset.classes)  # auto-detects number of classes

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze base layers
for param in model.parameters():
    param.requires_grad = False

# Replace final layer with correct number of output classes
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, NUM_CLASSES)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)

print(f"Model loaded and moved to: {device}")
print(f"Output classes: {NUM_CLASSES} → {train_dataset.classes}")

Model loaded and moved to: cpu
Output classes: 3 → ['ClassA-Drought', 'ClassB-PestInfestation', 'ClassC-Healthy']


In [4]:
# ── Cell 4: Loss Function & Optimizer ─────────────────────────────────
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

In [5]:
# ── Cell 5: Training Loop ──────────────────────────────────────────────
EPOCHS = 10

for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print("-" * 10)
    start_time = time.time()

    # --- PHASE 1: TRAINING ---
    model.train()
    running_train_loss = 0.0
    correct_train = 0
    total_train   = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train  += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_train_loss = running_train_loss / len(train_dataset)
    epoch_train_acc  = correct_train / total_train

    # --- PHASE 2: VALIDATION ---
    model.eval()
    running_val_loss = 0.0
    correct_val = 0
    total_val   = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss    = criterion(outputs, labels)
            running_val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val  += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    epoch_val_loss = running_val_loss / len(val_dataset)
    epoch_val_acc  = correct_val / total_val
    end_time = time.time()

    print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f}")
    print(f"Val Loss:   {epoch_val_loss:.4f} | Val Acc:   {epoch_val_acc:.4f}")
    print(f"Time: {end_time - start_time:.2f} seconds\n")

Epoch 1/10
----------
Train Loss: 0.5764 | Train Acc: 0.7823
Val Loss:   0.3127 | Val Acc:   0.8799
Time: 524.43 seconds

Epoch 2/10
----------
Train Loss: 0.2498 | Train Acc: 0.9334
Val Loss:   0.2310 | Val Acc:   0.9120
Time: 549.32 seconds

Epoch 3/10
----------


KeyboardInterrupt: 

In [ ]:
# ── Cell 6: Save the Model ─────────────────────────────────────────────
torch.save({
    "model_state_dict"    : model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "class_names"         : train_dataset.classes,  # FIX: was full_dataset.classes
}, "resnet18_final.pth")

print("✅ Model saved as resnet18_final.pth")

In [ ]:
# ── Cell 7: Test Set Evaluation ────────────────────────────────────────
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# FIX: Use correct path (same pattern as train/val)
test_dataset = datasets.ImageFolder(
    root=r"C:\Users\ACER\Desktop\rice_datasets\test",
    transform=test_val_transforms
)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
print(f"Test samples: {len(test_dataset)}")

# Collect predictions
all_preds  = []
all_labels = []

model.eval()
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs        = model(inputs)
        _, predicted   = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# FIX: Use train_dataset.classes (full_dataset doesn't exist)
class_names = train_dataset.classes

# Accuracy
test_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"\n✅ Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

# Classification Report
print("\n" + classification_report(all_labels, all_preds, target_names=class_names))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 8: Predict a Single Image ────────────────────────────────────
from PIL import Image

# ── Settings ───────────────────────────
MODEL_PATH  = "resnet18_final.pth"
CLASS_NAMES = train_dataset.classes          # auto-loaded from your dataset
IMAGE_PATH  = "test.png"                     # ← change to your image path
# ───────────────────────────────────────

# Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
checkpoint = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()
print("✅ Model loaded successfully!")

# Predict function
def predict_image(image_path, model, class_names, device):
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ])
    image  = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs    = model(tensor)
        probs      = torch.softmax(outputs, dim=1)[0]
        confidence, predicted = torch.max(probs, 0)

    predicted_class = class_names[predicted.item()]

    plt.figure(figsize=(5, 5))
    plt.imshow(image)
    plt.axis("off")
    plt.title(
        f"Predicted: {predicted_class.upper()}\n"
        f"Confidence: {confidence.item() * 100:.2f}%",
        fontsize=14, fontweight="bold"
    )
    plt.show()

    print("\n📊 All Probabilities:")
    for i, cls in enumerate(class_names):
        bar = "█" * int(probs[i].item() * 30)
        print(f"  {cls:<20} {probs[i].item()*100:5.2f}%  {bar}")

    return predicted_class, confidence.item()

# FIX: Removed stray ``` at end of function
predicted_class, confidence = predict_image(IMAGE_PATH, model, CLASS_NAMES, device)